# mjorbit P4 v3: bar-pose estimator on the realistic arm (arm2), fine-tuned from v2

v3 data: 125 episodes recorded on the arm2 camera scene (joint modules, new gripper look; same client, bar, sensors and randomisation as v2). Training starts from the v2 network (epoch 23, `pose_v2/best.pt`), scores it on the new data first (epoch 0), then fine-tunes for 10 epochs at a lower learning rate. Episodes 0-109 train, 110-124 select the checkpoint.

Before running: **Runtime > Change runtime type > T4 GPU**. Drive layout: `MyDrive/mjorbit_p4/data_v3/` (5 shards + SHA256SUMS), `MyDrive/mjorbit_p4/code_v3/` (scripts) and `MyDrive/mjorbit_p4/pose_v2/best.pt` (from the v2 run). Results go to `MyDrive/mjorbit_p4/pose_v3/`.

If the session disconnects, run all cells again: training resumes from the last finished epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/mjorbit_p4'
!ls -la {D} {D}/data_v3 {D}/code_v3
!ls -la {D}/pose_v2/best.pt

In [ ]:
# copy the shards to the local disk, check them, unpack (skipped if already packed in this session)
import os
if not os.path.exists('/content/demos_v3_pack/index.json'):
    !mkdir -p /content/shards /content/demos_v3
    !cp {D}/data_v3/* /content/shards/
    !cd /content/shards && sha256sum -c SHA256SUMS
    !for f in /content/shards/demos_v3_*.tar; do tar -xf $f -C /content/demos_v3 && rm $f; done
    !ls /content/demos_v3 | wc -l

In [ ]:
# pack into flat arrays (about 4 GB), one frame per image
!mkdir -p /content/code && cp {D}/code_v3/*.py /content/code/
if not os.path.exists('/content/demos_v3_pack/index.json'):
    !cd /content/code && python pack_data.py /content/demos_v3 /content/demos_v3_pack 6
    !rm -rf /content/demos_v3
!ls -la /content/demos_v3_pack

In [ ]:
# fine-tune from v2: epoch 0 scores the v2 weights on arm2 images, then 10 epochs at lr 3e-4
!mkdir -p {D}/pose_v3
!cd /content/code && python train_pose.py --pack /content/demos_v3_pack --out {D}/pose_v3 --init {D}/pose_v2/best.pt --epochs 10 --lr 3e-4 --n-train 110 --workers 4 --augment 1 2>&1 | tee -a {D}/pose_v3/train_colab.log

In [ ]:
# per-epoch validation error: median mm overall, settle, reach, approach; minutes (epoch 0 = v2 weights)
import json
for r in json.load(open(f'{D}/pose_v3/log.json')):
    print(r['epoch'], *(round(r[k]['pos_mm_med'], 1) for k in ('all', 'settle', 'reach', 'approach')), r['min'])